In [1]:
import librosa as lb
import numpy as np
import pickle
from eval_tools import getGroundTruthTimestamps
import utils.constants as constants
import plotly.graph_objs as go
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

### Load Alignments

In [8]:
s = 25

dtw_hyp_file = f"experiments/DTW/s{s}/hyp.npy"
noa_hyp_file = f"experiments/NOA/s{s}/hyp.npy"

# load
dtw_hyp = np.load(dtw_hyp_file)
noa_hyp = np.load(noa_hyp_file)

# load the ground truth timestamps
query_annot_file = f'scenarios/s{s}/query.beats'
ref_annot_file = f'scenarios/s{s}/ref.beats'
gt = getGroundTruthTimestamps(query_annot_file, ref_annot_file).T
gt = gt

In [4]:
pair_txt_file = f'scenarios/s{s}/pair.txt'
pair_txt = open(pair_txt_file, 'r').readlines()
pair_txt = [line.strip().split() for line in pair_txt]
id1 = pair_txt[0][0]
id2 = pair_txt[0][1]

# load the chroma features
audio_path_1 = f'Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/{id1}.wav'
audio_path_2 = f'Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/{id2}.wav'

# load the features
f1 = np.load(f'features/chroma_stft_norm2/{id1}.npy')
f2 = np.load(f'features/chroma_stft_norm2/{id2}.npy')

In [ ]:
from noa import alignNOA
path_noa = alignNOA(f1, f2)

In [ ]:
from OnlineAlignment.online_alignment import run_offline_oltw
path_oltw = np.flip(run_offline_oltw(f1, f2) * 512 / 22050, axis=0)

In [19]:
import plotly.graph_objs as go

# plot dtw_hyp, noa_hyp, and path_noa, and gt
fig = go.Figure()
fig.update_layout(
    width=700,
    height=700,
    xaxis_title="Query Time (seconds)",
    yaxis_title="Reference Time (seconds)",
    title="Alignment Paths Visualization"
)


# Add traces for dtw_hyp, noa_hyp, and path_noa
# fig.add_trace(go.Scatter(x=dtw_hyp[0], y=dtw_hyp[1], mode='lines', name='DTW Hyp'))
fig.add_trace(go.Scatter(x=noa_hyp[0], y=noa_hyp[1], mode='lines', name='NOA Hyp'))
fig.add_trace(go.Scatter(x=path_noa[0], y=path_noa[1], mode='lines', name='Updated NOA'))
fig.add_trace(go.Scatter(x=path_oltw[0], y=path_oltw[1], mode='lines', name='OLTW'))
fig.add_trace(go.Scatter(x=gt[0], y=gt[1], mode='markers', name='GT'))